# MEI editorial checks (optional page combine)

**Workflow 1 — handle MEI files.** Run CAMAT's automated editorial checks on one or more MEI files. Optionally join facsimile-linked **page** files into one score first, then check that combined file.

**Two paths (same notebook):**

| Mode | Set | Input | Output |
| --- | --- | --- | --- |
| **Check only** | `COMBINE_PAGES = False` | one page, many pages, or an already-combined `*_full.mei` / `*_corr.mei` | CSV/JSON reports under `TARGET_DIR` |
| **Combine then check** | `COMBINE_PAGES = True` | several `*_facs_zones.mei` pages that belong to one piece | prepared copies, `*_full.mei`, CSV/JSON reports |

The **combine** example uses ten Buxtehude pages in `test_corpus/buxtehude_pages/`. For **check only**, point `MEI_INPUTS` at a single file (for example a page right after IIIF integration) and set `COMBINE_PAGES = False`.

Generated files are written under `TARGET_DIR` (default: `converted_mei/consistency_tutorial/`). Source paths in `MEI_INPUTS` are not overwritten.

| Output | When |
| --- | --- |
| `input_consistency_report.csv` / `.json` | check-only mode |
| `page_consistency_report.csv` / `.json` | combine mode, per-page pass |
| `editorial_consistency_report.csv` or `{stem}_full_consistency_report.csv` | editorial suite (RELAX NG, Verovio, facsimile links, …) |
| `unique_ids/`, `noppq/` copies, `{stem}_full.mei` | combine mode only |

**Findings and `<annot>` export.** CAMAT integrates check results in two ways: **CSV/JSON reports** (default) and optional export into MEI `<annot>` elements (`ANNOTATE_COMBINED`). Our editorial workflow relies heavily on [mei-friend](https://mei-friend.mdw.ac.at/), which only displays a limited number of annotations per score (`annotationDisplayLimit`, default **100**, up to **300** in settings). A typical report has many more rows, so we work from the CSV/JSON files instead — they are easy to open, filter, and edit in a spreadsheet or notebook while you correct the MEI in mei-friend.

Remote `http(s)://` links in `MEI_INPUTS` are downloaded once and cached locally. Optional rewrite/cleanup flags live in [`mei_corrected_full_checks.ipynb`](../mei_corrected_full_checks.ipynb).

Set `RUN_PIPELINE = True` after reviewing the plan, then open the checked MEI in [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb).

In [1]:
# You can leave this cell unchanged.
try:
    from camat import (
        annotate_mei_from_report,
        annotation_filter_description,
        combine_meis,
        display_path,
        find_camat_root,
        load_report,
        prepare_pages_for_combine,
        report_summary,
        resolve_mei_inputs,
        resolve_repo_path,
        run_checker,
        run_editorial_checks,
    )
except ModuleNotFoundError:
    import setup_camat
    from camat import (
        annotate_mei_from_report,
        annotation_filter_description,
        combine_meis,
        display_path,
        find_camat_root,
        load_report,
        prepare_pages_for_combine,
        report_summary,
        resolve_mei_inputs,
        resolve_repo_path,
        run_checker,
        run_editorial_checks,
    )

## 1. Configure inputs and mode

`MEI_INPUTS` accepts repo-relative paths, absolute paths, directories, explicit file lists, or `http(s)://` links (cached locally on first use). Reports and any derived MEI are written under `TARGET_DIR`; files listed in `MEI_INPUTS` are not changed.

**Mode**

- `COMBINE_PAGES` — `False`: run checks on the listed file(s) as they are (one page, many pages, or a full score). `True`: prepare page copies, join them into one `*_full.mei`, then check that file.
- `RUN_PIPELINE` — master switch; copies, combined MEI, and reports stay off until this is `True`.

**Prepare and combine** (only when `COMBINE_PAGES = True`)

- `UNIQUE_XML_IDS` — copy each page with unique `xml:id` values so concatenated pages do not collide.
- `STRIP_PPQ` — strip import/playback `@ppq` and `@dur.ppq` from those copies. Written `@dur` / `@dots` stay.
- `STRIP_ACCID_GES` — strip leftover MusicXML `@accid.ges` from those copies. MEI uses `@accid` (written) for notation and reserves `@accid.ges` for performed inflection; MusicXML imports often duplicate both and can make MIDI playback disagree with Verovio.
- `RENUMBER_COMBINED_MEASURES` — number measures `1…n` across the whole score after joining.
- `NUMBER_COMBINED_MEASURE_ZONES` — number measure zones to match those measure numbers.
- `NORMALIZE_SINGLE_LAYER_NUMBERS` — if a staff has only one layer, set that layer `@n` to `1`.
- `KEEP_ORIGINAL_COMBINED_STAFF_NAMES` — keep staff labels from the first page’s `<scoreDef>` as the canonical names.
- `COMBINED_STAFF_NAMES_ONLY_AT_START` — remove staff labels from later `<scoreDef>` elements so names appear once at the start.
- `COMBINED_STAFF_NAMES` / `COMBINED_STAFF_ABBREVIATIONS` — optional overrides, e.g. `{"1": "Violino.", "2": "Viola da gamba."}`. Leave `None` to keep existing labels.
- `COMBINED_STAFF_GROUP_LABEL` / `COMBINED_STAFF_GROUP_ABBREVIATION` — optional label on the nested `<staffGrp>`.

**Checks** (all report-only; each input file when `COMBINE_PAGES = False`, the combined file when `True`)

- `CHECK_PPQ` — compare leftover `@dur.ppq` with written `@dur` / `@dots`.
- `CHECK_PUBLICATION_PROFILE` — header, identifiers, and facsimile page topology.
- `CHECK_RELAXNG` — validate against the packaged MEI 5.1 CMN schema (`xmllint` required).
- `CHECK_FB_TSTAMP` — figured-bass `<harm>` that still use `@startid` instead of `@tstamp` + `@staff`.
- `CHECK_PB_FACS` — `<pb>` `@facs` against the following measure’s facsimile surface.
- `CHECK_VEROVIO` — load the file with Verovio and collect warning/error logs.
- `VEROVIO_RENDER_PAGES` — also render every page (slower; catches layout-only issues). Needs `CHECK_VEROVIO`.
- `CHECK_IIIF_LINKS` — network check that each `<graphic @target>` is a reachable IIIF URL.

**`<annot>` export (integrated, off by default)**

CAMAT can write selected findings into `<annot>` on a new `*_annotated.mei` for in-editor review. Because mei-friend's annotation display cap is much smaller than a full check report, `ANNOTATE_COMBINED` stays `False` here and we use CSV/JSON reports as the main hand-off. Enable annotation only for a small, filtered error set you want to click through in mei-friend.

- `ANNOTATE_COMBINED` — write `<annot>` elements into `*_annotated.mei` (off by default).
- `ANNOTATE_SEVERITIES` — which report severities to attach, e.g. `{"error"}` or `{"error", "warning"}`.
- `ANNOTATE_CATEGORIES` / `ANNOTATE_CHECKS` — `None` means all; otherwise a set such as `{"rhythm"}` or `{"broken_internal_reference"}`.
- `ANNOTATE_EXCLUDE_CHECKS` — checks to skip, e.g. `{"measure_sequence"}`.
- `MAX_ANNOTATIONS` — export cap when annotation is enabled (default 100; mei-friend `annotationDisplayLimit` default is also 100, max 300).

Review the plan, then set `RUN_PIPELINE = True`.

In [ ]:
ROOT = find_camat_root()

# MEI file(s) to check. Each item may be a .mei file, a directory of *.mei files,
# or an http(s):// link (raw GitHub, MEI sample URL, …).
#
# Active example: directory of facsimile-linked page files (combine workflow).
MEI_INPUTS = ["test_corpus/buxtehude_pages"]
# Single local page after IIIF integration:
# MEI_INPUTS = ["converted_mei/iiif_tutorial/bsb00023199_00185_facs_zones.mei"]
# Already-combined score:
# MEI_INPUTS = ["converted_mei/consistency_tutorial/bsb00023199_00126_facs_zones_full.mei"]
# Explicit list of pages:
# MEI_INPUTS = [
#     "test_corpus/buxtehude_pages/bsb00023199_00126_facs_zones.mei",
#     "test_corpus/buxtehude_pages/bsb00023199_00127_facs_zones.mei",
# ]
# Remote MEI (downloads once; needs network on first run):
# MEI_INPUTS = [
#     "https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.0/Music/Complete_examples/Bach-JS_Ein_feste_Burg.mei",
# ]

# Reports and working copies land under TARGET_DIR.
TARGET_DIR = "converted_mei/consistency_tutorial"

# False = check MEI_INPUTS as-is. True = prepare page copies and join into *_full.mei first.
COMBINE_PAGES = False

# Prepare copies when combining. Originals in MEI_INPUTS stay unchanged.
UNIQUE_XML_IDS = True          # rename colliding xml:id values across pages
STRIP_PPQ = True              # drop import @ppq / @dur.ppq from the copies
STRIP_ACCID_GES = True        # drop leftover MusicXML @accid.ges from the copies

# How the joined score is numbered and labelled.
RENUMBER_COMBINED_MEASURES = True          # measures 1…n through the whole piece
NUMBER_COMBINED_MEASURE_ZONES = True       # matching numbers on measure zones
NORMALIZE_SINGLE_LAYER_NUMBERS = True      # lone layer on a staff becomes @n="1"
KEEP_ORIGINAL_COMBINED_STAFF_NAMES = True   # first page's scoreDef is the name source
COMBINED_STAFF_NAMES_ONLY_AT_START = True   # strip staff labels from later scoreDefs
# Optional overrides, e.g. {"1": "Violino.", "2": "Viola da gamba."}
COMBINED_STAFF_NAMES = None
COMBINED_STAFF_ABBREVIATIONS = None
COMBINED_STAFF_GROUP_LABEL = None
COMBINED_STAFF_GROUP_ABBREVIATION = None

# Editorial checks. All of these only write report rows, not the source MEI.
CHECK_PPQ = True                  # leftover @dur.ppq vs written @dur/@dots
CHECK_PUBLICATION_PROFILE = True  # header, identifiers, facsimile topology
CHECK_RELAXNG = True               # MEI 5.1 CMN schema via xmllint
CHECK_FB_TSTAMP = True            # figured-bass @startid that can be @tstamp+@staff
CHECK_PB_FACS = True             # <pb> @facs vs the next measure's surface
CHECK_VEROVIO = True             # Verovio load warnings/errors
VEROVIO_RENDER_PAGES = True      # also render every page (slower)
CHECK_IIIF_LINKS = False         # network check of <graphic @target> IIIF URLs

# Optional: export findings into <annot> on *_annotated.mei (see intro; CSV is default).
ANNOTATE_COMBINED = False
ANNOTATE_SEVERITIES = {"error"}   # e.g. {"error", "warning"}
ANNOTATE_CATEGORIES = None        # None = all; e.g. {"rhythm", "references"}
ANNOTATE_CHECKS = None            # None = all; e.g. {"broken_internal_reference"}
ANNOTATE_EXCLUDE_CHECKS = set()   # e.g. {"measure_sequence"}
MAX_ANNOTATIONS = 100            # export cap when ANNOTATE_COMBINED is True

# Reports stay off until this is True.
RUN_PIPELINE = False

## 2. Review the plan

This cell lists the resolved input files and the active mode. It does not write reports or change MEI.

In [3]:
target_dir = resolve_repo_path(TARGET_DIR, repo_root=ROOT)
page_files = resolve_mei_inputs(MEI_INPUTS, ROOT)
if not page_files:
    raise FileNotFoundError("No .mei files found in MEI_INPUTS")

print(f"Inputs:     {len(page_files)}")
for path in page_files:
    print(f"  {display_path(path, repo_root=ROOT)}")
print(f"Target dir: {display_path(target_dir, repo_root=ROOT)}")
print(f"Mode:       {'combine then check' if COMBINE_PAGES else 'check only'}")
print(f"Run enabled: {RUN_PIPELINE}")
if ANNOTATE_COMBINED:
    print("Annotation filter:", annotation_filter_description(
        severities=ANNOTATE_SEVERITIES,
        categories=ANNOTATE_CATEGORIES,
        checks=ANNOTATE_CHECKS,
        exclude_checks=ANNOTATE_EXCLUDE_CHECKS,
        max_annotations=MAX_ANNOTATIONS,
    ))
else:
    print("Findings: CSV/JSON under TARGET_DIR (ANNOTATE_COMBINED is False)")

Pages:      10
  test_corpus/buxtehude_pages/bsb00023199_00126_facs_zones.mei
  test_corpus/buxtehude_pages/bsb00023199_00127_facs_zones.mei
  test_corpus/buxtehude_pages/bsb00023199_00128_facs_zones.mei
  test_corpus/buxtehude_pages/bsb00023199_00129_facs_zones.mei
  test_corpus/buxtehude_pages/bsb00023199_00130_facs_zones.mei
  test_corpus/buxtehude_pages/bsb00023199_00131_facs_zones.mei
  test_corpus/buxtehude_pages/bsb00023199_00132_facs_zones.mei
  test_corpus/buxtehude_pages/bsb00023199_00133_facs_zones.mei
  test_corpus/buxtehude_pages/bsb00023199_00134_facs_zones.mei
  test_corpus/buxtehude_pages/bsb00023199_00135_facs_zones.mei
Target dir: converted_mei/consistency_tutorial
Annotation filter: severities=['error'], categories=all, checks=all, exclude_checks=none, max_annotations=100
Run enabled: True


## 3. Run checks

With `RUN_PIPELINE = True`, this cell writes CSV/JSON reports under `TARGET_DIR` and prints a short summary. Open the CSV files directly, or load them elsewhere, to work through findings.

**Check only** (`COMBINE_PAGES = False`): general consistency on each input, then the full editorial suite on the same file(s).

**Combine then check** (`COMBINE_PAGES = True`): prepared page copies → per-page report → `*_full.mei` → editorial CSV/JSON report.

In [4]:
if not RUN_PIPELINE:
    print("Skipped. Review the plan, then set RUN_PIPELINE = True.")
    df = combined_df = None
    checked_paths = []
    combine_result = None
else:
    target_dir.mkdir(parents=True, exist_ok=True)
    checked_paths = page_files
    combine_result = None

    if COMBINE_PAGES:
        prepared = prepare_pages_for_combine(
            page_files,
            target_dir,
            unique_xml_ids=UNIQUE_XML_IDS,
            strip_ppq=STRIP_PPQ,
            strip_accid_ges=STRIP_ACCID_GES,
        )
        print(prepared.message)
        print(f"Prepared {len(prepared.files)} page file(s).")
        checker_inputs = prepared.files
        page_csv = target_dir / "page_consistency_report.csv"
    else:
        checker_inputs = page_files
        page_csv = target_dir / "input_consistency_report.csv"

    run_checker(
        checker_inputs,
        root=ROOT,
        csv_out=page_csv,
        json_out=page_csv.with_suffix(".json"),
        check_ppq=CHECK_PPQ,
        publication_profile=CHECK_PUBLICATION_PROFILE,
    )
    df = load_report(page_csv)
    print(report_summary(df).to_string(index=False))

    if COMBINE_PAGES:
        base_stem = prepared.files[0].stem.removesuffix("_noppq").removesuffix("_unique_ids")
        combined_path = target_dir / f"{base_stem}_full.mei"
        combine_result = combine_meis(
            prepared.files,
            combined_path,
            renumber_measures=RENUMBER_COMBINED_MEASURES,
            number_measure_zones=NUMBER_COMBINED_MEASURE_ZONES,
            normalize_single_layer_numbers=NORMALIZE_SINGLE_LAYER_NUMBERS,
            staff_names=COMBINED_STAFF_NAMES,
            staff_abbreviations=COMBINED_STAFF_ABBREVIATIONS,
            staff_group_label=COMBINED_STAFF_GROUP_LABEL,
            staff_group_abbreviation=COMBINED_STAFF_GROUP_ABBREVIATION,
            keep_original_staff_names=KEEP_ORIGINAL_COMBINED_STAFF_NAMES,
            staff_names_only_at_start=COMBINED_STAFF_NAMES_ONLY_AT_START,
        )
        checked_paths = [combine_result.path]
        print(f"Combined {len(prepared.files)} page(s) into {display_path(combine_result.path, repo_root=ROOT)}")
        print(f"Duplicate xml:id values: {len(combine_result.duplicate_ids)}")
        print(f"Renumbered measures: {combine_result.renumbered_measures}")
        combined_csv = combine_result.path.with_name(f"{combine_result.path.stem}_consistency_report.csv")
    else:
        combined_csv = target_dir / "editorial_consistency_report.csv"

    combined_df = run_editorial_checks(
        checked_paths,
        root=ROOT,
        csv_out=combined_csv,
        json_out=combined_csv.with_suffix(".json"),
        check_ppq=CHECK_PPQ,
        publication_profile=CHECK_PUBLICATION_PROFILE,
        check_relaxng=CHECK_RELAXNG,
        check_fb_tstamp=CHECK_FB_TSTAMP,
        check_pb_facs=CHECK_PB_FACS,
        check_verovio=CHECK_VEROVIO,
        verovio_render_pages=VEROVIO_RENDER_PAGES,
        check_iiif_links=CHECK_IIIF_LINKS,
    )
    print(report_summary(combined_df).to_string(index=False))

    if ANNOTATE_COMBINED and len(checked_paths) == 1:
        source_path = checked_paths[0]
        annotated_path = source_path.with_name(f"{source_path.stem}_annotated.mei")
        annotation_result = annotate_mei_from_report(
            source_path,
            combined_df,
            annotated_path,
            severities=ANNOTATE_SEVERITIES,
            categories=ANNOTATE_CATEGORIES,
            checks=ANNOTATE_CHECKS,
            exclude_checks=ANNOTATE_EXCLUDE_CHECKS,
            max_annotations=MAX_ANNOTATIONS,
        )
        print(f"Annotated MEI: {display_path(annotation_result.path, repo_root=ROOT)}")
        print(f"Created {annotation_result.created} annotation(s); skipped {annotation_result.skipped}.")
    elif ANNOTATE_COMBINED:
        print("Skipped annotation: ANNOTATE_COMBINED needs one checked file (check only with one input, or combine pages).")

Renamed 42 duplicate xml:id value(s). Removed 7480 @ppq/@dur.ppq and @accid.ges attribute(s).
Prepared 10 page file(s).
Checked 10 MEI file(s). Wrote 1586 findings to /home/egor/Nextcloud/code/public_repos/camat_v2/converted_mei/consistency_tutorial/page_consistency_report.csv.
Findings by severity: error=257, warning=280, info=1049
severity    category                         check  count
   error   facsimile          measure_page_context    227
   error   facsimile        page_break_per_surface     10
   error publication                cmn_relaxng_pi     10
   error publication             cmn_schematron_pi     10
    info      rhythm               rhythm_no_meter    804
    info    layering single_layer_not_numbered_one    215
    info publication        application_provenance     19
    info   facsimile           surface_iiif_canvas     10
    info       terms corpus_term_spelling_variants      1
 warning      rhythm      layer_duration_underfull     48
 warning      rhythm       

## Notes

1. Uncomment one `MEI_INPUTS` example above, or point it at your own file, folder, or URL.
2. Use `COMBINE_PAGES = False` to check without joining pages.
3. Reports land under `TARGET_DIR`; combine mode also writes `unique_ids/`, `noppq/`, and `*_full.mei` there.
4. Review findings in the CSV/JSON reports under `TARGET_DIR`, correct the MEI in your editor, then re-run until the report is clean.
5. Turn `CHECK_IIIF_LINKS` on only when you want a network check of facsimile URLs.
6. `CHECK_RELAXNG` needs `xmllint`. Install it if that pass raises.
7. Open the checked or combined MEI in [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb).
8. Optional file rewrites (`CLEAN_*`, `FIX_*`) belong in [`mei_corrected_full_checks.ipynb`](../mei_corrected_full_checks.ipynb).